In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
# %matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames


In [ ]:
%matplotlib widget

In [ ]:
output_dir = "/mnt/SpatialSequenceLearning/Simon/hmm_actions"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="INFO")


In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)[0]
session_names = fullfnames2snames(session_dirs)

In [ ]:
framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names[:7],)
# framew_beh = analytics.get_analytics('BehaviorFramewise', session_names=session_names,)
framew_beh

In [ ]:
# s_id = '2024-12-11_17-42' #17
# trial_id = 34
# # time_col = 'frame_pc_timestamp'
# time_col = 'frame_ephys_timestamp'

# session_data = framew_beh.xs(s_id, level='session_id')
# trial_data = session_data.loc[session_data['trial_id'] == trial_id]

# trial_data = trial_data.set_index(time_col, drop=True, append=False)
# trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
# trial_data

In [ ]:

def plot_one_trial(trial_data, trial_actions, s_id, trial_id, cluster_labels=None, n_clusters=None):
    """
    Plot a single trial with behavioral metrics and events.
    
    Parameters:
    -----------
    trial_data : pd.DataFrame
        Trial data with time index
    s_id : str
        Session ID
    trial_id : int
        Trial ID
    cluster_labels : array-like, optional
        GMM cluster assignments for each frame (same length as trial_data)
    n_clusters : int, optional
        Number of clusters (required if cluster_labels provided)
    """
    
    # Create figure with optional cluster subplot
    if cluster_labels is not None:
        fig, (ax_clusters, ax_main) = plt.subplots(
            2, 1, figsize=(0.025*len(trial_data), 10), 
            height_ratios=[1, 4],
            sharex=True
        )
        plt.subplots_adjust(hspace=0.05)
    else:
        fig, ax_main = plt.subplots(figsize=(0.025*len(trial_data), 10))

    # main_metric = 'frame_velocity'
    # main_metric = 'frame_raw'
    # main_metric = 'frame_RawYawPitch_abs_vel_sum'

    # which cue was shown
    color = 'orange' if trial_data['cue'].iloc[0] == 1 else 'purple'

    # cue limits
    cue_entry_time = trial_data[(trial_data['frame_position'] >-80) & 
                                (trial_data['frame_position'] <25)].index.values
    ax_main.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.1)
    # cue limits
    cue_entry_time = trial_data[(trial_data['track_zone'].isin(['visibleCue', 'nextToCue']))].index.values
    ax_main.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.1)

    # R1 limits
    R1_entry_time = trial_data[trial_data['track_zone']=='reward1Zone'].index.values
    ax_main.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.2)
    
    # R2 limits
    R2_entry_time = trial_data[trial_data['track_zone']=='reward2Zone'].index.values
    ax_main.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.5)

    r1_thr = (trial_data['velocity_threshold_at_R1'] * trial_data['fps'])
    r1_frames = r1_thr[(r1_thr.index>R1_entry_time[0]) & (r1_thr.index<R1_entry_time[-1])].index
    ax_main.plot(r1_thr, linestyle='--', color='black', alpha=.4, label='R1 thr.')
    
    r2_thr = (trial_data['velocity_threshold_at_R2'] * trial_data['fps'])
    r2_frames = r2_thr[(r2_thr.index>R2_entry_time[0]) & (r2_thr.index<R2_entry_time[-1])].index
    ax_main.plot(r2_thr.loc[r2_frames], linestyle='--', color='black', alpha=.4, label='R2 thr.')

    # events
    event_labels = {
        'reward-sound_count': {'label': 'Sound', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': '>'},
        'reward-valve-open_count': {'label': 'Reward', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': 'o'},
        'lick_count': {'label': 'Lick', 'color': 'black', 'size': 50, 'alpha': 0.4, 'marker': '|'},
        'reward-removed_count': {'label': 'Reward-removed', 'color': 'red', 'size': 80, 'alpha': 0.7, 'marker': 'x'}
    }
    
    for event in event_labels.keys():
        event_data = trial_data[trial_data[event] >= 1]
        if len(event_data) == 0:
            continue
        
        ax_main.scatter(event_data.index, event_data['frame_RawYawPitch_abs_vel_sum_500msMedian'],
                       color=event_labels[event]['color'], s=event_labels[event]['size'],
                       marker=event_labels[event]['marker'], label=event_labels[event]['label'],
                       alpha=event_labels[event]['alpha'], zorder=5)

    

    # ball_forward_vel	
    # ball_forward_acc	
    # ball_rotation_vel	
    # ball_forward+rotation_vel	
    # ball_forward+rotation_acc	
    # licking	
    # forward_proportion	
    # forward_vs_rotation_corr	
    # forward_std_500ms	
    # rotation_std_500ms
    
    ax_main.plot(trial_actions['ball_forward_vel'], zorder=1, color='blue', label='ball-forward')
    ax_main.plot(trial_actions['ball_forward_acc'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-acc')
    ax_main.plot(trial_actions['ball_rotation_vel'], zorder=1, color='purple', label='ball-rotation')
    ax_main.plot(trial_actions['ball_forward+rotation_vel'], zorder=1, color='darkblue', label='ball-forward+rotation')
    ax_main.plot(trial_actions['ball_forward+rotation_acc'], zorder=1, color='darkblue', alpha=0.4, linestyle=':', label='ball-forward+rotation-acc')
    ax_main.plot(trial_actions['forward_proportion']*100, zorder=1, color='orange', alpha=0.7, linestyle='-', label='forward proportion (%)')
    ax_main.plot(trial_actions['forward_vs_rotation_corr']*100, zorder=1, color='green', alpha=0.7, linestyle='-', label='forward vs rotation corr (%)')
    ax_main.plot(trial_actions['forward_std_500ms']*10, zorder=1, color='cyan', alpha=0.3, linestyle='-', label='forward std (500ms)')
    ax_main.plot(trial_actions['rotation_std_500ms']*10, zorder=1, color='magenta', alpha=0.3, linestyle='-', label='rotation std (500ms)')

    # # main metrics
    # ax_main.plot(actions[''], zorder=1, color='blue', label='ball-summed')
    # ax_main.plot(trial_data['frame_RawYawPitch_abs_vel_sum_500msMedian'], zorder=1, color='darkblue', alpha=0.5, linestyle='-', label='ball-summed=smoothed')
    # # ax_main.plot(trial_data['frame_pitch'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-sideways')
    # ax_main.plot(trial_data['frame_YawPitch_abs_vel_500msMedian'].abs(), zorder=1, color='purple', alpha=0.3, linestyle=':', label='ball-rotation-smoothed')
    # ax_main.plot(trial_data['frame_YawPitch_abs_acc_500msMedian'], zorder=1, color='purple', alpha=0.4, linestyle=':', label='ball-rotation-sm-acc')
    
    # ax_main.plot(trial_data['frame_raw_500msMedian'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-forward-smoothed')
    # ax_main.plot(trial_data['frame_raw_acc_500msMedian'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-sm-acc')
    # # ax_main.plot(trial_data['facecam_pose_nose_neck_body1_angle'].rolling(6).median(), zorder=1, color='darkred', linestyle=':', label='head-angle')

    ax_main.set_xlabel('Time (s)')
    ax_main.set_ylabel('Velocity [cm/s]')
    ax_main.set_title(f'Session {s_id} - Trial {trial_id}')
    ax_main.legend(ncol=2, fontsize=8)
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)

    # =========================================================================
    # Cluster visualization subplot (if labels provided)
    # =========================================================================
    if cluster_labels is not None:
        times = trial_data.index.values
        cluster_colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
        
        # Use scatter plot - one point per frame, colored by cluster
        for c in range(-1, n_clusters):
            mask = cluster_labels == c
            if not mask.any():
                continue
            
            if c == -1:
                color = 'lightgray'
                y_val = -0.3
            else:
                color = cluster_colors[c]
                y_val = c
            
            ax_clusters.scatter(
                times[mask], 
                np.full(mask.sum(), y_val),
                c=[color], 
                s=200,
                marker='|',
                linewidths=1,
            )
        
        # Format cluster axis
        ax_clusters.set_ylim(-0.5, n_clusters - 0.5)
        ax_clusters.set_yticks(range(n_clusters))
        ax_clusters.set_yticklabels([f'C{c}' for c in range(n_clusters)], fontsize=8)
        ax_clusters.set_ylabel('Cluster', fontsize=9)
        ax_clusters.spines['top'].set_visible(False)
        ax_clusters.spines['right'].set_visible(False)
        ax_clusters.spines['bottom'].set_visible(False)
        ax_clusters.tick_params(bottom=False)
        
        # Add zone shading to cluster plot too
        ax_clusters.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.2)
        ax_clusters.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.1)
        ax_clusters.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.2)

    plt.tight_layout()
    # plt.show()
    return fig

In [ ]:
framew_beh

In [ ]:
# from scipy import stats

# def calculate_rolling_metrics(var1, var2, window_size, debug=True, n_windows=5, p_threshold=0.05):
#     """
#     Calculates rolling correlation and standard deviation for two variables.

#     Args:
#         var1 (pd.Series): The first variable.
#         var2 (pd.Series): The second variable.
#         window_size (int): The size of the rolling window.
#         debug (bool): If True, plots scatter plots with fitted correlation lines and r values.
#         n_windows (int): Grid size for debug plots (e.g., 5 for 5x5 grid).
#         p_threshold (float): P-value threshold for statistical significance.

#     Returns:
#         tuple: A tuple containing:
#             - pd.Series: Rolling correlation between var1 and var2.
#             - pd.Series: Rolling standard deviation of var1.
#             - pd.Series: Rolling standard deviation of var2.
#     """
#     rolling_corr = var1.rolling(window=window_size, center=True).corr(var2).clip(-1, 1) **5
    
#     if debug:
#         fig, axes = plt.subplots(nrows=n_windows, ncols=n_windows, figsize=(12, 12))
#         axes = axes.flatten()
        
#         # Sample n_windows^2 evenly spaced windows from the data
#         total_windows = len(var1) - window_size + 1
#         window_indices = np.linspace(0, total_windows - 1, n_windows * n_windows, dtype=int)
        
#         for idx, win_idx in enumerate(window_indices):
#             x_window = var1.iloc[win_idx:win_idx + window_size].values
#             y_window = var2.iloc[win_idx:win_idx + window_size].values
            
#             # valid_mask = ~(np.isnan(x_window) | np.isnan(y_window))
#             x_valid = x_window#[valid_mask]
#             y_valid = y_window#[valid_mask]
            
#             ax = axes[idx]
            
#             if len(x_valid) > 2:
#                 ax.scatter(x_valid, y_valid, alpha=0.6, s=20, color='blue')
                
#                 # Pearson correlation with p-value
#                 r, p_value = stats.pearsonr(x_valid, y_valid)
                
#                 # Calculate R-squared (coefficient of determination)
#                 r_squared = r ** 3
                
#                 # Calculate variance ratio (how much x variance explains y variance)
#                 x_var = np.var(x_valid)
#                 y_var = np.var(y_valid)
#                 var_ratio = x_var / (y_var + 1e-10)  # avoid division by zero
                
#                 # Fit line only if p-value is significant
#                 if r**3 > .3:
#                     z = np.polyfit(x_valid, y_valid, 1)
#                     p = np.poly1d(z)
#                     x_line = np.linspace(x_valid.min(), x_valid.max(), 50)
#                     ax.plot(x_line, p(x_line), 'g--', linewidth=2, label='Fit')
#                     color = 'k'
#                 else:
#                     color = 'k'
                
#                 # Title with multiple metrics
#                 title = f'r={r:.2f} | R**5={r**5:.2f}\np={p_value:.4f}'
#                 ax.set_title(title, fontsize=8, color=color, fontweight='bold')
                
#                 ax.tick_params(labelsize=6)
#                 ax.grid(True, alpha=0.2)
#             else:
#                 ax.text(0.5, 0.5, 'Insufficient\ndata', ha='center', va='center', fontsize=8)
#                 ax.set_title('N/A', fontsize=8)
        
#         plt.tight_layout()
#         plt.show()
    
#     return rolling_corr

In [ ]:
action_cols = [
               # forward velocity and acc
               'frame_raw_500msMedian', 
               'frame_raw_acc_500msMedian', 
               # off rotations velocity
               'frame_YawPitch_abs_vel_500msMedian', 
               # sum for reward threshold and acc
               'frame_RawYawPitch_abs_vel_sum_500msMedian',
               'frame_RawYawPitch_abs_acc_sum_500msMedian',
               # others
               'lick_detected', 
               'trial_id',
            #    'facecam_pose_nose_neck_body1_angle', 'facecam_pose_nose_neck_body1_angle_velocity'
               ]

renamer = {
      'frame_raw_500msMedian': 'ball_forward_vel',
      'frame_raw_acc_500msMedian': 'ball_forward_acc',
      'frame_YawPitch_abs_vel_500msMedian': 'ball_rotation_vel',
      'frame_RawYawPitch_abs_vel_sum_500msMedian': 'ball_forward+rotation_vel',
      'frame_RawYawPitch_abs_acc_sum_500msMedian': 'ball_forward+rotation_acc',
      # 'frame_YawPitch_abs_acc_500msMedian': 'ball_rotation_acc',
      'lick_detected': 'licking',
}
corr, std1, std2 = calculate_rolling_metrics(framew_beh['frame_raw_500msMedian'], 
                                             framew_beh['frame_YawPitch_abs_vel_500msMedian'], 
                                             window_size=30) # 500ms window at 60fps = 30 frames


actions = framew_beh[action_cols].dropna()
actions = actions.rename(columns=renamer).drop(columns=['trial_id']) # just for NaN dropping (first ~80 frames of a session)

still_mask = actions['ball_forward+rotation_vel'] > 2.5
actions['forward_proportion'] = actions['ball_forward_vel'].abs()[still_mask] / (actions['ball_forward+rotation_vel'])[still_mask]
# centered, if animal still we assume forward and rotation are equally 
# important when animal is still, since we can't really say if it's a forward or rotation action when the animal is not moving
actions.loc[~still_mask, 'forward_proportion'] = .5 


actions['forward_vs_rotation_corr'] = corr.fillna(0)
# actions.loc[~still_mask,'forward_vs_rotation_corr'] = 0
actions['forward_std_500ms'] = std1
actions['rotation_std_500ms'] = std2



actions

actions.isna().sum()

In [ ]:

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(6, 6), )#sharex=True)
axes = axes.flatten()

actions_scaled = (actions - actions.mean()) / actions.std()

for i, col in enumerate(actions.columns):
    # standardize
    # val = (actions[col] - actions[col].mean()) / actions[col].std()
    val = actions_scaled[col]
    axes[i].hist(val, bins=30)
    axes[i].set_title(col, fontsize=6)
    # axes[i].set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# plot correlation matrix
plt.figure(figsize=(8, 6))
corr = actions.corr()

# # cluster correlation matrix using linked hierarchical clustering
# from scipy.cluster.hierarchy import linkage, leaves_list
# linked = linkage(corr, method='ward')
# leaves = leaves_list(linked)
# corr = corr.iloc[leaves, :].iloc[:, leaves]

im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Matrix of Action Features')
plt.tight_layout()
plt.show()

In [ ]:
# from sklearn.ensemble import IsolationForest

# iso = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
# outlier_mask = iso.fit_predict(actions) == -1  # -1 = outlier
# actions = actions.loc[~outlier_mask]
# print(f"Removed {outlier_mask.sum()} outliers from {len(actions)} frames")

In [ ]:
plt.close('all')

s_id = 4
s_id = framew_beh.index.unique('session_id')[s_id]
time_col = 'frame_pc_timestamp'

session_data = framew_beh.loc[framew_beh.index.get_level_values('session_id') == s_id]
for trial_id in session_data['trial_id'].dropna().unique().astype(int):
    if trial_id == -1:
        continue
    trial_data = session_data.loc[session_data['trial_id'] == trial_id]
    
    trial_frame_ids = trial_data.index.values
    trial_data = trial_data.set_index(time_col, drop=False, append=False)
    trial_actions = actions.loc[actions.index.isin(trial_frame_ids)]
    trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    trial_actions.index = trial_data.index
    
    
    fig = plot_one_trial(trial_data, trial_actions, s_id, trial_id)
    plt.show()
    
    if trial_id == 10:
        break  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm


def plot_joint_grid(
    df,
    max_points=2000,
    jitter_std=0.01,
    bins=30,
    figsize_per_dim=2.4,
    scatter_alpha=0.25,
    corr_circle_size=500,
):
    """
    Plot a joint distribution grid with marginals (top row and left column),
    and a Pearson correlation circle inside every joint plot.
    """

    df = df.copy()
    cols = df.columns
    n = len(cols)

    # ---- Determine which columns need jitter ----
    needs_jitter = {
        col: df[col].value_counts(dropna=False).shape[0] < 5
        for col in cols
    }

    # ---- Precompute jittered data ----
    df_jittered = df.copy()
    for col, apply_jitter in needs_jitter.items():
        if apply_jitter:
            col_range = df[col].max() - df[col].min()
            scale = jitter_std * col_range if col_range > 0 else jitter_std
            df_jittered[col] = df[col] + np.random.randn(len(df)) * scale

    # ---- Precompute correlation matrix ----
    corr = df.corr(method="pearson")

    # ---- Colormap ----
    corr_cmap = cm.get_cmap("RdYlGn")
    norm = plt.Normalize(vmin=-1, vmax=1)

    # ---- Figure setup ----
    fig, axes = plt.subplots(
        nrows=n,
        ncols=n,
        figsize=(figsize_per_dim * n, figsize_per_dim * n),
    )

    # ---- Plot ----
    for i, row_col in enumerate(cols):
        for j, col_col in enumerate(cols):
            ax = axes[i, j]

            if i == 0 and j > 0:
                # Top row: marginal distributions
                data = df[col_col].dropna()
                unique_vals = data.nunique()
                hist_bins = min(bins, unique_vals) if unique_vals > 1 else 1

                ax.hist(
                    data,
                    bins=hist_bins,
                    density=True,
                    color="gray",
                    alpha=0.7,
                )
                ax.set_title(col_col, fontsize=10)
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)

            elif j == 0 and i > 0:
                # Left column: marginal distributions
                data = df[row_col].dropna()
                unique_vals = data.nunique()
                hist_bins = min(bins, unique_vals) if unique_vals > 1 else 1

                ax.hist(
                    data,
                    bins=hist_bins,
                    density=True,
                    orientation="horizontal",
                    color="gray",
                    alpha=0.7,
                )
                ax.set_ylabel(row_col, fontsize=10)
                ax.set_xticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)

            else:
                # Joint scatter plot (includes first column for i>0,j>0 is false now fixed)
                if i > 0 and j > 0:
                    sub = df_jittered[[col_col, row_col]].dropna()
                    if len(sub) > max_points:
                        sub = sub.sample(max_points, random_state=42)

                    ax.scatter(
                        sub[col_col],
                        sub[row_col],
                        s=10,
                        alpha=scatter_alpha,
                        linewidths=0,
                    )

                    # ---- Correlation circle ----
                    r = corr.loc[row_col, col_col]
                    circle_color = corr_cmap(norm(r)) if not np.isnan(r) else "lightgray"

                    ax.scatter(
                        0.88,
                        0.88,
                        s=corr_circle_size,
                        c=[circle_color],
                        edgecolors="black",
                        transform=ax.transAxes,
                        zorder=5,
                        clip_on=False,
                    )
                    if not np.isnan(r):
                        ax.text(
                            0.88,
                            0.88,
                            f"{r:.2f}",
                            ha="center",
                            va="center",
                            fontsize=8,
                            fontweight="bold",
                            transform=ax.transAxes,
                            zorder=6,
                            clip_on=False,
                        )
                else:
                    # Top-left corner (i==0,j==0)
                    ax.axis("off")

            # ---- Axis labels ----
            if i == n - 1:
                ax.set_xlabel(col_col)
            else:
                ax.set_xlabel("")

            if j == 0 and i > 0:
                # ylabel already set for marginal
                pass
            elif j == 0:
                ax.set_ylabel("")
            else:
                ax.set_ylabel("")

    plt.tight_layout()
    return fig, axes


In [ ]:
fig, axes = plot_joint_grid(actions_scaled)
plt.show()
# actions_scaled

In [ ]:
# fit umap embedding on actions
import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def fit_umap_and_pca(actions, max_n=200_000):
    """
    Fit UMAP and PCA on actions data. If data is larger than max_n,
    fits on a subsample but transforms ALL data.
    
    Returns:
        umap_embedding: shape (n_samples, 3) - embedding for ALL actions
        pca_embedding: shape (n_samples, n_components) - embedding for ALL actions
        scaler: fitted StandardScaler
        reducer: fitted UMAP reducer
        pca: fitted PCA
    """
    # Standardize ALL data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(actions)
    n_total = data_scaled.shape[0]
    
    # Subsample for fitting if needed
    if n_total > max_n:
        print(f"Subsampling data for UMAP/PCA fitting: {n_total} -> {max_n}")
        random_indices = np.random.choice(n_total, size=max_n, replace=False)
        data_scaled_fit = data_scaled[random_indices, :]
    else:
        data_scaled_fit = data_scaled
        random_indices = np.arange(n_total)
        

    # PCA for comparison
    print("\n=== PCA Analysis ===")
    pca = PCA(n_components=min(60, actions.shape[1]))
    pca.fit(data_scaled_fit)  # Fit on subsample
    pca_embedding = pca.transform(data_scaled)  # Transform ALL data

    print(f"PCA explained variance ratio (first 3 components): {pca.explained_variance_ratio_[:3]}")
    print(f"PCA cumulative variance (first 3 components): {np.cumsum(pca.explained_variance_ratio_[:3])[-1]:.3f}")
    print(f"PCA cumulative variance (all components): {np.cumsum(pca.explained_variance_ratio_)}")

    # UMAP embedding
    print("\n=== UMAP Analysis ===")
    print("Fitting UMAP (this may take a few minutes)...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=30,
        min_dist=0.1,
        metric='euclidean',
        random_state=42,
        verbose=True
    )
    reducer.fit(data_scaled_fit)  # Fit on subsample
    print("Transforming all data with fitted UMAP...")
    umap_embedding = reducer.transform(data_scaled)  # Transform ALL data
    print(f"UMAP complete! Shape: {umap_embedding.shape}")
    
    return umap_embedding, pca_embedding, scaler, reducer, pca, random_indices

In [ ]:
from plotly.subplots import make_subplots
from plotly import graph_objects as go

def plot_umap_pca_colored_by(umap_embedding, pca_embedding, labels,
                             output_subdir='umap_action', max_points=10_000, label_name='Cluster'):
    """
    Create UMAP and PCA 3D visualizations colored by each feature or by provided labels.
    
    Automatically converts categorical/string labels to integers for plotting.
    
    Parameters:
    -----------
    output_subdir : str
        Subdirectory for saving HTML files
    max_points : int
        Maximum points to plot (subsamples if larger)
    labels : array-like
        Plot colored by these labels. Can be numeric or categorical (strings).
        Must have same length as embeddings.
    label_name : str
        Name for the colorbar when using labels (default: 'Cluster')
    """
    import os
    import pandas as pd
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    assert len(labels) == len(umap_embedding), "Labels length must match number of data points"
    
    # Convert labels to array if needed
    labels_array = np.array(labels) if not isinstance(labels, np.ndarray) else labels
    
    # Detect categorical labels and convert to integers
    label_mapping = None
    if labels_array.dtype == object or labels_array.dtype.kind in ['U', 'S']:  # string types
        print(f"Detected categorical labels, converting to integers...")
        unique_labels = pd.Series(labels_array).unique()
        label_mapping = {label: i for i, label in enumerate(unique_labels)}
        labels_numeric = np.array([label_mapping[lbl] for lbl in labels_array])
        print(f"Label mapping: {label_mapping}")
    else:
        labels_numeric = labels_array
    
    # Subsample indices if needed
    n_total = len(umap_embedding)
    if n_total > max_points:
        print(f"Subsampling {n_total:,} -> {max_points:,} points")
        sample_idx = np.random.choice(n_total, size=max_points, replace=False)
    else:
        sample_idx = np.arange(n_total)
    
    # Get color values and create hover text
    color_values = labels_numeric[sample_idx]
    # Use original labels for hover text if categorical
    hover_labels = labels_array[sample_idx] if label_mapping else color_values
    hover_text = [f'{label_name}: {v}' for v in hover_labels]
    
    # Configure colorbar and colorscale based on data type
    if label_mapping:
        # Categorical: use discrete colorscale
        colorscale = 'plotly3'  # Qualitative colorscale for categories
        # Get unique categories in display order (by numeric code)
        unique_codes = sorted(set(label_mapping.values()))
        code_to_label = {v: k for k, v in label_mapping.items()}
        colorbar_tickvals = unique_codes
        colorbar_ticktext = [code_to_label[c] for c in unique_codes]
        colorbar_dict = dict(
            title=label_name, 
            x=0.46,
            tickvals=colorbar_tickvals,
            ticktext=colorbar_ticktext,
            tickmode='array'
        )
    else:
        # Numeric: use continuous colorscale
        colorscale = 'Viridis'
        colorbar_dict = dict(title=label_name, x=0.46)
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f'UMAP - {label_name}', f'PCA - {label_name}'),
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        horizontal_spacing=0.05
    )
    
    # UMAP scatter
    fig.add_trace(go.Scatter3d(
        x=umap_embedding[sample_idx, 0],
        y=umap_embedding[sample_idx, 1],
        z=umap_embedding[sample_idx, 2],
        mode='markers',
        marker=dict(size=2, color=color_values, colorscale=colorscale, 
                    showscale=True, colorbar=colorbar_dict, opacity=0.6),
        text=hover_text,
        hovertemplate='<b>%{text}</b><br>UMAP: %{x:.2f}, %{y:.2f}, %{z:.2f}<extra></extra>'
    ), row=1, col=1)
    
    # PCA scatter
    fig.add_trace(go.Scatter3d(
        x=pca_embedding[sample_idx, 0],
        y=pca_embedding[sample_idx, 1],
        z=pca_embedding[sample_idx, 2],
        mode='markers',
        marker=dict(size=2, color=color_values, colorscale=colorscale, showscale=False, opacity=0.6),
        text=hover_text,
        hovertemplate='<b>%{text}</b><br>PC: %{x:.2f}, %{y:.2f}, %{z:.2f}<extra></extra>'
    ), row=1, col=2)
    
    fig.update_layout(title_text=f'Embedding colored by {label_name}', height=600, width=1400, showlegend=False)
    fig.update_scenes(camera=dict(eye=dict(x=1.5, y=1.5, z=1.3)))
    
    html_file = f"{viz_dir}{label_name.replace('/', '_').replace(' ', '_')}_umap_pca.html"
    fig.write_html(html_file)
    print(f"✓ Saved: {html_file}")
    return viz_dir

In [ ]:
umap_embedding, pca_embedding, scaler, reducer, pca, random_indices = fit_umap_and_pca(actions, max_n=200_00)
print(f"Actions shape: {actions.shape}")
print(f"UMAP embedding shape: {umap_embedding.shape}")
print(f"PCA embedding shape: {pca_embedding.shape}")

In [ ]:
actions

In [ ]:

for i, s_id in enumerate(actions.index.get_level_values('session_id').unique()):
    indices = actions.index.get_level_values('session_id') == s_id
    # print(f"Plotting session {s_id} (code {s_id_int}) with {indices.sum()} frames")
    
    # Pass categorical labels directly - function will convert to int internally
    labels = framew_beh.loc[actions.index, 'frame_position'][indices].values
    plot_umap_pca_colored_by(umap_embedding[indices], pca_embedding[indices], 
                             labels, output_subdir='sessions', 
                             label_name=f'position_umap_pca_session{i:02d}', max_points=100_000)

In [ ]:
# feature-wise plotting

for feature in actions.columns:
    print(f"Plotting UMAP/PCA colored by: {feature}")
    plot_umap_pca_colored_by(umap_embedding, pca_embedding, actions[feature].values, 
                             label_name=feature, max_points=100_000)

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

def fit_gmm(embedding, n_components_range=range(2, 15), max_n=200_000):
    """
    Fit GMM on embedding and select optimal number of components using BIC.
    
    If embedding is larger than max_n, fits on a subsample but predicts labels for ALL data.
    Labels are sorted by occurrence: 0 = most common cluster, 1 = second most common, etc.
    
    Parameters:
    -----------
    embedding : array-like
        Embedding data to cluster (n_samples, n_features)
    n_components_range : range
        Range of component numbers to try
    max_n : int
        Maximum number of samples to use for fitting (subsamples if larger).
        Labels are still predicted for ALL data.
        
    Returns:
    --------
    gmm : fitted GaussianMixture model
    labels : cluster assignments for ALL data (sorted by occurrence: 0=most common)
    results : dict with BIC/AIC scores, silhouette scores, and cluster_counts
    """
    X = embedding
    n_total = X.shape[0]
    
    # Subsample for fitting if needed
    if n_total > max_n:
        print(f"Subsampling for GMM fitting: {n_total:,} -> {max_n:,}")
        fit_idx = np.random.choice(n_total, size=max_n, replace=False)
        X_fit = X[fit_idx]
    else:
        X_fit = X
        fit_idx = np.arange(n_total)
    
    print(f"Fitting GMM on embedding (fit shape: {X_fit.shape}, total: {X.shape})")
    
    bics, aics, silhouettes = [], [], []
    models = []
    
    for n in n_components_range:
        gmm = GaussianMixture(n_components=n, covariance_type='full', 
                              random_state=42, n_init=3)
        gmm.fit(X_fit)
        models.append(gmm)
        
        bics.append(gmm.bic(X_fit))
        aics.append(gmm.aic(X_fit))
        
        labels_fit = gmm.predict(X_fit)
        sil = silhouette_score(X_fit, labels_fit) if n > 1 else 0
        silhouettes.append(sil)
        
        print(f"  n={n:2d}: BIC={bics[-1]:,.0f}, AIC={aics[-1]:,.0f}, silhouette={sil:.3f}")
    
    # Select best model by BIC
    best_idx = np.argmin(bics)
    best_n = list(n_components_range)[best_idx]
    best_gmm = models[best_idx]
    
    # Predict labels for ALL data using fitted model
    print(f"Predicting labels for all {n_total:,} samples...")
    raw_labels = best_gmm.predict(X)
    
    # =========================================================================
    # Sort clusters by occurrence: 0 = most common, 1 = second most common, etc.
    # =========================================================================
    raw_counts = pd.Series(raw_labels).value_counts().sort_values(ascending=False)
    raw_cluster_order = raw_counts.index.tolist()  # original IDs ordered by count
    
    # Create mapping: original cluster ID -> new rank (0=most common)
    old_to_new = {orig: rank for rank, orig in enumerate(raw_cluster_order)}
    
    # Remap labels
    labels = np.array([old_to_new[l] for l in raw_labels])
    
    # Recompute counts with new labels (for verification)
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    
    print(f"\n✓ Best model: n_components={best_n} (lowest BIC)")
    print(f"  Cluster sizes (sorted): {cluster_counts.to_dict()}")
    
    results = {
        'n_components_range': list(n_components_range),
        'bics': bics,
        'aics': aics,
        'silhouettes': silhouettes,
        'best_n': best_n,
        'cluster_counts': cluster_counts,  # already in order 0,1,2,...
        'old_to_new_mapping': old_to_new,
    }
    
    return best_gmm, labels, results

In [ ]:
def plot_gmm_results(gmm, labels, results, output_subdir='gmm_plots', max_points=10_000):
    """
    Visualize GMM clustering results.
    
    Creates:
    1. Model selection plot (BIC/AIC/silhouette curves)
    2. 3D scatter of clusters on UMAP and PCA
    3. Cluster feature profiles (z-scored bar plots per cluster)
    
    NOTE: Labels are expected to be already sorted by occurrence from fit_gmm
          (0 = most common cluster, 1 = second most common, etc.)
    """
    import os
    import plotly.express as px
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_clusters = results['best_n']
    
    # Labels are already sorted by occurrence (0=most common) from fit_gmm
    # Just compute counts for display
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    
    # Subsample if needed
    n_total = len(labels)
    if n_total > max_points:
        sample_idx = np.random.choice(n_total, size=max_points, replace=False)
    else:
        sample_idx = np.arange(n_total)
    
    # =========================================================================
    # 1. Model selection plot (BIC/AIC/Silhouette)
    # =========================================================================
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    n_range = results['n_components_range']
    
    axes[0].plot(n_range, results['bics'], 'b-o', label='BIC')
    axes[0].axvline(results['best_n'], color='r', linestyle='--', label=f'Best: {results["best_n"]}')
    axes[0].set_xlabel('Number of components')
    axes[0].set_ylabel('BIC')
    axes[0].set_title('BIC (lower is better)')
    axes[0].legend()
    
    axes[1].plot(n_range, results['aics'], 'g-o', label='AIC')
    axes[1].axvline(results['best_n'], color='r', linestyle='--')
    axes[1].set_xlabel('Number of components')
    axes[1].set_ylabel('AIC')
    axes[1].set_title('AIC (lower is better)')
    
    axes[2].plot(n_range, results['silhouettes'], 'm-o', label='Silhouette')
    axes[2].axvline(results['best_n'], color='r', linestyle='--')
    axes[2].set_xlabel('Number of components')
    axes[2].set_ylabel('Silhouette Score')
    axes[2].set_title('Silhouette (higher is better)')
    
    plt.suptitle(f'GMM Model Selection on UMAP', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}gmm_model_selection.png", dpi=150)
    plt.show()
    
    # =========================================================================
    # 2. 3D scatter colored by cluster (interactive plotly)
    # =========================================================================
    plot_umap_pca_colored_by(
        umap_embedding, pca_embedding,
        output_subdir=output_subdir, 
        max_points=max_points, 
        labels=labels, 
        label_name=f'GMM k={n_clusters}'
    )
    
    # =========================================================================
    # 3. Cluster feature profiles (z-scored means, sorted by contribution)
    # =========================================================================
    features = actions.columns.tolist()
    n_features = len(features)
    
    # Compute z-scored cluster means (how each cluster deviates from overall mean)
    global_mean = actions.mean()
    global_std = actions.std()
    
    # Build profiles for each cluster (already in occurrence order: 0=most common)
    cluster_profiles = pd.DataFrame(index=features, columns=range(n_clusters))
    for c in range(n_clusters):
        cluster_mask = labels == c
        cluster_mean = actions[cluster_mask].mean()
        # Z-score: how many std devs away from global mean
        cluster_profiles[c] = (cluster_mean - global_mean) / global_std
    
    # Create subplot for each cluster - horizontal bar plot
    n_cols = min(4, n_clusters)
    n_rows = int(np.ceil(n_clusters / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 0.5*n_features*n_rows + 1))
    axes = np.atleast_1d(axes).flatten()
    
    cluster_colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
    
    for c in range(n_clusters):
        ax = axes[c]
        count = cluster_counts[c]
        
        # Use same feature order as heatmap (no sorting) for consistency
        z_scores = cluster_profiles[c].values.astype(float)
        
        # Color bars: green for positive, red for negative
        bar_colors = ['#2ecc71' if z > 0 else '#e74c3c' for z in z_scores]
        
        y_pos = np.arange(n_features)
        ax.barh(y_pos, z_scores, color=bar_colors, edgecolor='none', alpha=0.8)
        ax.axvline(0, color='black', linewidth=0.8)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(features, fontsize=8)
        ax.invert_yaxis()  # top feature at top
        ax.set_xlabel('Z-score (std devs from mean)', fontsize=9)
        ax.set_title(f'C{c} (n={count})', fontsize=11, 
                     color=cluster_colors[c], fontweight='bold')
        ax.set_xlim(-max(3, np.abs(z_scores).max()*1.1), max(3, np.abs(z_scores).max()*1.1))
        ax.grid(axis='x', alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Hide unused axes
    for j in range(n_clusters, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'Cluster Feature Profiles (k={n_clusters}) - Ordered by Occurrence - Green=HIGH, Red=LOW', fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}gmm_cluster_profiles.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 4. Heatmap overview: all clusters x features
    # =========================================================================
    fig, ax = plt.subplots(figsize=(n_clusters + 2, n_features * 0.4 + 1))
    
    profile_matrix = cluster_profiles.values.astype(float)
    im = ax.imshow(profile_matrix, aspect='auto', cmap='RdYlGn', 
                   vmin=-2.5, vmax=2.5)
    
    ax.set_xticks(range(n_clusters))
    ax.set_xticklabels([f'C{c}' for c in range(n_clusters)], fontsize=10)
    ax.set_yticks(range(n_features))
    ax.set_yticklabels(features, fontsize=9)
    ax.set_xlabel('Cluster (ordered by occurrence)')
    ax.set_ylabel('Feature')
    
    # Add text annotations
    for i in range(n_features):
        for j in range(n_clusters):
            val = profile_matrix[i, j]
            color = 'white' if abs(val) > 1.5 else 'black'
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', 
                   fontsize=8, color=color)
    
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Z-score', fontsize=10)
    
    plt.title(f'Cluster Feature Composition Heatmap (k={n_clusters})', fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}gmm_cluster_heatmap.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 5. Cluster summary statistics
    # =========================================================================
    print(f"\n=== Cluster Summary (k={n_clusters}) - Ordered by Occurrence ===")
    print(f"{'Cluster':<10} {'Count':<10} {'%':<8} {'Top 3 HIGH features':<50} {'Top 3 LOW features':<50}")
    for c in range(n_clusters):
        count = cluster_counts[c]
        pct = count / len(labels) * 100
        
        z_scores = cluster_profiles[c].sort_values()
        top_high = ', '.join([f"{f}(+{z:.1f})" for f, z in z_scores.tail(3)[::-1].items()])
        top_low = ', '.join([f"{f}({z:.1f})" for f, z in z_scores.head(3).items()])
        
        print(f"C{c:<9} {count:<10} {pct:>5.1f}%   {top_high:<50} {top_low:<50}")
        
    # # ============================================================================
    # # Cluster Interpretation (k=9) - Based on feature profiles
    # # ============================================================================

    # cluster_descriptions_k9 = {
    #     0: {'name': 'Fast Forward + Left Turn', 'description': 'High velocity(+0.5σ), fast forward(+0.4σ). Strong left rotation: pitch(-1.2σ), yaw(-0.8σ). Head down(-0.7σ).'},
    #     1: {'name': 'Sprint + Yaw Acceleration', 'description': 'Fastest forward(+1.7σ), high velocity(+1.5σ), yaw accel(+0.6σ). Low pitch accel(-0.7σ), minimal licking(-0.3σ).'},
    #     2: {'name': 'Right Turn / Orienting', 'description': 'Right rotation: yaw(+0.7σ), pitch(+0.5σ). Slower forward(-0.6σ), low yaw accel(-0.5σ). Head down(-0.4σ).'},
    #     3: {'name': 'Head Up / Alert', 'description': 'Head elevated(+1.2σ), forward accel(+0.6σ), slight right yaw(+0.3σ). Low licking(-0.3σ). Stable head velocity.'},
    #     4: {'name': 'Slow / Hesitating', 'description': 'Low velocity(-0.9σ), slow forward(-0.8σ). Slight right pitch(+0.4σ), yaw accel(+0.4σ), left yaw(-0.6σ).'},
    #     5: {'name': 'Stationary / Head Down', 'description': 'Very low velocity(-1.1σ), slow forward(-1.0σ). Head down(-1.4σ). Slight right pitch(+0.6σ). Minimal movement.'},
    #     6: {'name': 'Licking / Reward', 'description': 'Extremely high licking(+2.7σ). Right rotation: yaw(+1.0σ), pitch(+0.6σ). Low forward(-0.2σ). Reward consumption.'},
    #     7: {'name': 'Active Head Scanning', 'description': 'Fast head movement(+1.7σ), head up(+1.3σ). Low locomotion: velocity(-0.5σ), forward(-0.5σ), left yaw(-0.4σ).'},
    #     8: {'name': 'Freeze / Head Still', 'description': 'Frozen head(-3.3σ velocity). Head down(-0.8σ), slight left yaw(-0.4σ). Near-zero ball movement. Vigilance/hesitation.'},
    # }

    # # Print formatted summary
    # print("\n" + "=" * 80)
    # print(f"CLUSTER INTERPRETATIONS (k={n_clusters}) - Ordered by Occurrence")
    # print("=" * 80)
    # for c in range(min(n_clusters, len(cluster_descriptions_k9))):
    #     if c in cluster_descriptions_k9:
    #         desc = cluster_descriptions_k9[c]
    #         count = cluster_counts[c]
    #         pct = count / len(labels) * 100
    #         print(f"\n🔹 C{c}: {desc['name']} ({pct:.1f}% of frames)")
    #         print(f"   {desc['description']}")
    # print("\n" + "=" * 80)
    
    print(f"\n✓ All plots saved to: {viz_dir}")
    return viz_dir, cluster_profiles

In [ ]:
# Fit GMM and visualize
gmm, labels, results = fit_gmm(umap_embedding, n_components_range=[4], max_n=100_000)
plot_gmm_results(gmm, labels, results)

In [ ]:
cluster_assign = pd.DataFrame({
    'gmm_k4': labels,
}, index=actions.index)
cluster_assign


In [ ]:
# def get_counts(arr):
#     """Get run lengths of consecutive identical values in an array."""
#     if len(arr) == 0: 
#         return np.array([])
#     # Find where the value changes (current element != next element)
#     locs = np.where(arr[1:] != arr[:-1])[0]
#     # Add boundaries (the start and end of the array)
#     boundaries = np.concatenate(([-1], locs, [len(arr) - 1]))
#     # Calculate the difference between consecutive boundary indices
#     return np.diff(boundaries)


# def plot_state_dynamics(embedding, labels, title='State', 
#                         embedding_name='Embedding', frame_duration_ms=16.6,
#                         save_path=None):
#     """
#     Plot diagnostic visualizations for state/cluster dynamics.
    
#     Parameters:
#     -----------
#     embedding : array-like (n_samples, n_features)
#         Embedding data to compute distances (UMAP, PCA, or raw features).
#         Euclidean distance between consecutive frames is computed.
#     labels : array-like (n_samples,)
#         State/cluster assignments for each frame (e.g., from GMM or HMM).
#     title : str
#         Title prefix for the plots (e.g., 'GMM k=5', 'HMM k=9').
#     embedding_name : str
#         Name of the embedding for axis labels (e.g., 'UMAP', 'PCA', 'Features').
#     frame_duration_ms : float
#         Duration of each frame in milliseconds (default: 16.6ms = 60fps).
#     save_path : str, optional
#         If provided, saves the figure to this path.
        
#     Returns:
#     --------
#     fig : matplotlib Figure
#     stats : dict with computed statistics
#     """
#     plt.close('all')
    
#     # Ensure labels is a numpy array
#     labels = np.asarray(labels)
#     embedding = np.asarray(embedding)
    
#     assert len(labels) == len(embedding), \
#         f"Labels ({len(labels)}) and embedding ({len(embedding)}) must have same length"

#     # Create a 1x3 figure layout
#     fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
#     # =============================================================================
#     # Panel 1: Embedding distances between consecutive points
#     # =============================================================================
#     ax = axes[0]
#     distances = np.linalg.norm(np.diff(embedding, axis=0), axis=1)
#     ax.hist(distances, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
#     ax.set_yscale('log')

#     # Percentile lines with annotations
#     p99 = np.percentile(distances, 99)
#     p90 = np.percentile(distances, 90)
#     p50 = np.percentile(distances, 50)

#     ax.axvline(p99, color='red', linestyle='--', linewidth=2, label=f'99th: {p99:.2f}')
#     ax.axvline(p90, color='orange', linestyle='--', linewidth=2, label=f'90th: {p90:.2f}')
#     ax.axvline(p50, color='green', linestyle='--', linewidth=2, label=f'median: {p50:.2f}')

#     ax.set_xlabel(f'{embedding_name} distance', fontsize=11)
#     ax.set_ylabel('Count (log)', fontsize=11)
#     ax.set_title(f'{embedding_name} Distances Between\nConsecutive Frames ({frame_duration_ms:.1f}ms)', 
#                  fontsize=12, fontweight='bold')
#     ax.legend(fontsize=9, loc='upper right')
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

#     # Add text annotation for mean
#     ax.text(0.95, 0.75, f'Mean: {distances.mean():.2f}\nStd: {distances.std():.2f}', 
#             transform=ax.transAxes, ha='right', va='top', fontsize=10,
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

#     # =============================================================================
#     # Panel 2: Proportion of state switches
#     # =============================================================================
#     ax = axes[1]
#     state_switches = np.diff(labels) != 0
#     prop_switches = state_switches.sum() / len(state_switches)

#     # Create a pie chart (donut) for visualization
#     switch_data = [prop_switches, 1 - prop_switches]
#     colors = ['#e74c3c', '#2ecc71']
#     pie_labels = [f'Switch\n({prop_switches:.1%})', f'Stay\n({1-prop_switches:.1%})']
#     wedges, texts, autotexts = ax.pie(switch_data, colors=colors, labels=pie_labels, 
#                                        autopct='', startangle=90, 
#                                        wedgeprops=dict(edgecolor='white', linewidth=2))

#     # Add center circle for donut effect
#     centre_circle = plt.Circle((0, 0), 0.5, fc='white')
#     ax.add_artist(centre_circle)

#     # Center text
#     ax.text(0, 0, f'Switch Rate\n{prop_switches:.1%}', ha='center', va='center', 
#             fontsize=14, fontweight='bold')
#     ax.set_title(f'{title} State Transitions\n(Frame-to-Frame)', fontsize=12, fontweight='bold')

#     # =============================================================================
#     # Panel 3: State duration (n-frames in same state)
#     # =============================================================================
#     ax = axes[2]
#     section_lengths = get_counts(labels)

#     # Calculate statistics
#     prop_length_1 = (section_lengths > 1).sum() / len(section_lengths)
#     prop_length_20 = (section_lengths > 20).sum() / len(section_lengths)
#     median_length = np.median(section_lengths)
#     mean_length = np.mean(section_lengths)

#     ax.hist(section_lengths, bins=100, color='purple', edgecolor='none', alpha=0.8)
#     ax.set_yscale('log')

#     # Threshold lines
#     threshold_frames = 20
#     threshold_ms = threshold_frames * frame_duration_ms
#     ax.axvline(threshold_frames, color='orange', linestyle='--', linewidth=2, 
#                label=f'>{threshold_frames} frames (>{threshold_ms:.0f}ms): {prop_length_20:.1%}')
#     ax.axvline(median_length, color='green', linestyle='-', linewidth=2,
#                label=f'Median: {median_length:.0f} frames')

#     ax.set_xlabel('State duration (frames)', fontsize=11)
#     ax.set_ylabel('Count (log)', fontsize=11)
#     ax.set_title(f'{title} State Duration\n(Consecutive Frames in Same State)', 
#                  fontsize=12, fontweight='bold')
#     ax.legend(fontsize=9, loc='upper right')
#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)

#     # Add text annotation
#     ax.text(0.95, 0.55, 
#             f'Mean: {mean_length:.1f} frames\n({mean_length*frame_duration_ms:.0f} ms)\n\nN segments: {len(section_lengths):,}',
#             transform=ax.transAxes, ha='right', va='top', fontsize=10,
#             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

#     plt.suptitle(f'{title} Dynamics Analysis', fontsize=14, fontweight='bold', y=0.98)
#     plt.tight_layout()
    
#     if save_path:
#         plt.savefig(save_path, dpi=150, bbox_inches='tight')
#         print(f"✓ Saved: {save_path}")
    
#     plt.show()
    
#     # Compile statistics
#     stats = {
#         'embedding_name': embedding_name,
#         'n_samples': len(labels),
#         'n_states': len(np.unique(labels)),
#         'switch_rate': prop_switches,
#         'mean_state_duration_frames': mean_length,
#         'mean_state_duration_ms': mean_length * frame_duration_ms,
#         'median_state_duration_frames': median_length,
#         'median_state_duration_ms': median_length * frame_duration_ms,
#         'prop_segments_gt_1_frame': prop_length_1,
#         'prop_segments_gt_20_frames': prop_length_20,
#         'distance_mean': distances.mean(),
#         'distance_std': distances.std(),
#         'distance_median': p50,
#         'distance_p90': p90,
#         'distance_p99': p99,
#     }
    
#     print(f"\n=== {title} Dynamics Summary ===")
#     print(f"Switch rate: {prop_switches:.1%}")
#     print(f"Median state duration: {median_length:.0f} frames ({median_length*frame_duration_ms:.0f} ms)")
#     print(f"Mean state duration: {mean_length:.1f} frames ({mean_length*frame_duration_ms:.0f} ms)")
#     print(f"Prop of segments >1 frame: {prop_length_1:.1%}")
#     print(f"Prop of segments >20 frames: {prop_length_20:.1%}")
    
#     return fig, stats

In [ ]:
# # Example usage with GMM labels
# fig, gmm_stats = plot_state_dynamics(
#     embedding=umap_embedding,
#     labels=cluster_assign['gmm_k4'].values,
#     title='GMM k=4',
#     embedding_name='UMAP',
#     frame_duration_ms=16.6,
#     save_path=f"{output_dir}/gmm_k4_dynamics.png"
# )



In [ ]:
actions_scaled

In [ ]:
actions

In [ ]:
from hmmlearn.hmm import GaussianHMM
from sklearn.model_selection import KFold
import random

# Set global random seed for reproducibility
RANDOM_SEED = 42

def _set_all_seeds(seed):
    """Set all random seeds for full reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    # Note: hmmlearn uses numpy internally

_set_all_seeds(RANDOM_SEED)

def fit_hmm_models(X, lengths=None, n_components_range=range(2, 15), covariance_type='full', 
                   n_iter=200, n_init=3, n_cv_folds=2, random_state=RANDOM_SEED, verbose=False):
    """
    Fit HMM models for a range of component numbers.
    
    Returns all fitted models and selection metrics. Model selection (best_idx) 
    is done separately in plotting/analysis functions.
    
    Parameters:
    -----------
    X : array-like, shape (n_samples, n_features)
        Training data
    lengths : array-like or None
        Lengths of sequences for HMM fitting
    n_components_range : range
        Range of component numbers to try
    covariance_type : str
        'full', 'diag', 'spherical', 'tied'
    n_iter : int
        Max EM iterations per initialization
    n_init : int
        Number of random initializations to try. The best model (highest LL) is kept.
        This helps avoid local optima. Default=3.
    n_cv_folds : int
        Number of cross-validation folds
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Whether to print detailed fitting progress (default=False)
        
    Returns:
    --------
    fitting_results : dict with models, metrics, and metadata
    """
    # CRITICAL: Reset ALL seeds at the start for full reproducibility
    _set_all_seeds(random_state)
    
    print(f"Fitting HMM models on data (shape: {X.shape})")
    print(f"  Covariance type: {covariance_type}, Max iterations: {n_iter}, N initializations: {n_init}")
    print(f"  CV folds: {n_cv_folds}, Random state: {random_state}")
    
    log_likelihoods, aics, bics, cv_scores, cv_stds, models = [], [], [], [], [], []
    n_samples, n_features = X.shape
    kfold = KFold(n_splits=n_cv_folds, shuffle=False)  # shuffle=False for reproducibility with sequences
    
    for n in n_components_range:
        print(f"  n={n:2d}: ", end='')
        
        # Try multiple initializations and keep the best one
        best_model = None
        best_ll = -np.inf
        
        for init_idx in range(n_init):
            # Reset seed for this initialization
            _set_all_seeds(random_state + init_idx * 100)
            
            model = GaussianHMM(n_components=n, covariance_type=covariance_type, 
                               n_iter=n_iter, random_state=random_state + init_idx * 100, 
                               verbose=verbose)
            model.fit(X, lengths=lengths) if lengths is not None else model.fit(X)
            
            ll = model.score(X)
            if ll > best_ll:
                best_ll = ll
                best_model = model
                if verbose:
                    print(f"    Init {init_idx+1}/{n_init}: LL={ll:,.0f} (new best)")
            elif verbose:
                print(f"    Init {init_idx+1}/{n_init}: LL={ll:,.0f}")
        
        models.append(best_model)
        log_likelihoods.append(best_ll)
        
        # Compute parameters for AIC/BIC
        if covariance_type == 'full':
            n_cov_params = n * n_features * (n_features + 1) // 2
        elif covariance_type == 'diag':
            n_cov_params = n * n_features
        elif covariance_type == 'spherical':
            n_cov_params = n
        else:
            n_cov_params = n_features * (n_features + 1) // 2
        
        n_params = n * (n - 1) + (n - 1) + n * n_features + n_cov_params
        aic = -2 * best_ll + 2 * n_params
        bic = -2 * best_ll + n_params * np.log(n_samples)
        aics.append(aic)
        bics.append(bic)
        
        # Cross-validation - reset seed before CV for reproducibility
        _set_all_seeds(random_state + 1000)  # Different seed for CV to avoid correlation
        fold_scores = []
        for train_idx, test_idx in kfold.split(X):
            model_cv = GaussianHMM(n_components=n, covariance_type=covariance_type, 
                                   n_iter=n_iter, random_state=random_state, verbose=False)
            model_cv.fit(X[train_idx])
            fold_scores.append(model_cv.score(X[test_idx]))
        cv_scores.append(np.mean(fold_scores))
        cv_stds.append(np.std(fold_scores))
        
        print(f"LL={best_ll:,.0f}, AIC={aic:,.0f}, BIC={bic:,.0f}, CV-LL={cv_scores[-1]:,.0f} (±{cv_stds[-1]:,.0f})")
    
    print(f"\n✓ Fitted {len(models)} models (n_components: {list(n_components_range)}, {n_init} inits each)")
    print(f"  Best by BIC: n={list(n_components_range)[np.argmin(bics)]}")
    
    fitting_results = {
        'models': models,
        'n_components_range': list(n_components_range),
        'log_likelihoods': log_likelihoods,
        'aics': aics,
        'bics': bics,
        'cv_scores': cv_scores,
        'cv_stds': cv_stds,
        'covariance_type': covariance_type,
        'n_samples': n_samples,
        'n_features': n_features,
        'random_state': random_state,
    }
    return fitting_results


def select_and_decode_hmm(X, fitting_results, best_idx=None, sort_by_occurrence=False):
    """
    Select a model and decode states.
    
    Parameters:
    -----------
    X : array-like
        Data used for prediction
    fitting_results : dict
        Output from fit_hmm_models
    best_idx : int or None
        Index into n_components_range. If None, uses argmin(BIC).
    sort_by_occurrence : bool, default=False
        If True, reorder states so that state 0 is the most common, state 1 is 
        second most common, etc. If False, keep original HMM state ordering.
        
    Returns:
    --------
    labels : array, state assignments
    results : dict with model info, parameters, counts
    
    Note:
    -----
    If sort_by_occurrence=True, all returned parameters (means, covars, 
    transition_matrix, startprob) are remapped so that state 0 is the most common.
    If sort_by_occurrence=False, original HMM state ordering is preserved.
    """
    if best_idx is None:
        best_idx = np.argmin(fitting_results['bics'])
    
    n_components_range = fitting_results['n_components_range']
    best_n = n_components_range[best_idx]
    best_model = fitting_results['models'][best_idx]
    raw_labels = best_model.predict(X)
    
    if sort_by_occurrence:
        # Sort states by occurrence: 0 = most common
        raw_counts = pd.Series(raw_labels).value_counts().sort_values(ascending=False)
        raw_state_order = raw_counts.index.tolist()  # e.g., [3, 1, 0, 2] means original state 3 is most common
        old_to_new = {orig: rank for rank, orig in enumerate(raw_state_order)}  # {3: 0, 1: 1, 0: 2, 2: 3}
        labels = np.array([old_to_new[l] for l in raw_labels])
        
        # Remap transition matrix: new_trans[new_i, new_j] = old_trans[old_i, old_j]
        transition_matrix = best_model.transmat_.copy()
        remapped_trans = np.zeros_like(transition_matrix)
        for new_i in range(best_n):
            for new_j in range(best_n):
                old_i = raw_state_order[new_i]
                old_j = raw_state_order[new_j]
                remapped_trans[new_i, new_j] = transition_matrix[old_i, old_j]
        
        # Remap means, covariances, and start probabilities
        remapped_means = best_model.means_[raw_state_order]
        remapped_covars = best_model.covars_[raw_state_order]
        remapped_startprob = best_model.startprob_[raw_state_order]
        
        print(f"  State reordering (by occurrence): {old_to_new}")
    else:
        # Keep original ordering
        labels = raw_labels
        raw_state_order = list(range(best_n))  # [0, 1, 2, ...] - identity mapping
        old_to_new = {i: i for i in range(best_n)}
        remapped_trans = best_model.transmat_.copy()
        remapped_means = best_model.means_.copy()
        remapped_covars = best_model.covars_.copy()
        remapped_startprob = best_model.startprob_.copy()
    
    # Compute counts using the labels
    state_counts = pd.Series(labels).value_counts().sort_index()
    
    switches = np.diff(labels) != 0
    switch_rate = switches.sum() / len(switches)
    
    print(f"✓ Selected model: n_components={best_n} (index={best_idx})")
    print(f"  Sort by occurrence: {sort_by_occurrence}")
    print(f"  State sizes: {state_counts.to_dict()}")
    print(f"  State switch rate: {switch_rate:.3f}")
    
    results = {
        # From fitting
        'n_components_range': n_components_range,
        'log_likelihoods': fitting_results['log_likelihoods'],
        'aics': fitting_results['aics'],
        'bics': fitting_results['bics'],
        'cv_scores': fitting_results['cv_scores'],
        'cv_stds': fitting_results['cv_stds'],
        'covariance_type': fitting_results['covariance_type'],
        'random_state': fitting_results.get('random_state', 42),
        # Selected model
        'best_n': best_n,
        'best_idx': best_idx,
        'sort_by_occurrence': sort_by_occurrence,
        'state_counts': state_counts,
        'transition_matrix': remapped_trans,
        'startprob': remapped_startprob,
        'means': remapped_means,
        'covars': remapped_covars,
        'switch_rate': switch_rate,
        'old_to_new_mapping': old_to_new,
        'raw_state_order': raw_state_order,
    }
    return labels, results


# Curated color palette - visually distinct, aesthetically pleasing
STATE_COLORS = [
    '#E63946',  # Red (vibrant)
    '#457B9D',  # Steel blue
    '#2A9D8F',  # Teal
    '#E9C46A',  # Saffron/gold
    '#9B5DE5',  # Purple
    '#F77F00',  # Orange
    '#06D6A0',  # Mint green
    '#EF476F',  # Pink
    '#118AB2',  # Cerulean
    '#073B4C',  # Dark teal
    '#8338EC',  # Violet
    '#FB5607',  # Bright orange
    '#3A86FF',  # Dodger blue
    '#FFBE0B',  # Amber
    '#FF006E',  # Magenta
]


def get_state_colors(n_states):
    """Get consistent colors for states from curated palette."""
    if n_states <= len(STATE_COLORS):
        return STATE_COLORS[:n_states]
    cmap = plt.cm.get_cmap('Set2')
    return [cmap(i / n_states) for i in range(n_states)]


def style_axis(ax, grid=True, spines=False):
    """Apply consistent styling: remove spines, add grid."""
    if not spines:
        for spine in ax.spines.values():
            spine.set_visible(False)
    if grid:
        ax.grid(True, alpha=0.7, linestyle='-', linewidth=0.5)
        ax.set_axisbelow(True)


def plot_model_parameters(results, feature_names, output_subdir='hmm_plots'):
    """
    Visualize the Gaussian HMM model parameters for a specific model.
    
    A Gaussian HMM with K states has:
    - K emission distributions: N(μ_k, Σ_k) where μ_k is (n_features,) and Σ_k is (n_features, n_features)
    - Transition matrix A: (K, K) where A[i,j] = P(state_j | state_i)
    - Start probabilities π: (K,)
    
    This creates a dense visualization showing:
    - Row 1: Mean vectors for each state (as bar plots)
    - Row 2: Covariance matrices for each state (as heatmaps)
    - Row 3: Transition matrix and start probabilities
    
    Note: All parameters are already remapped so state 0 = most common, etc.
    """
    import os
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = results['best_n']
    means = results['means']  # (n_states, n_features) - already remapped
    covars = results['covars']  # (n_states, n_features, n_features) - already remapped
    trans_matrix = results['transition_matrix']  # already remapped
    startprob = results['startprob']  # already remapped
    state_counts = results['state_counts']  # indexed by new state (0, 1, 2, ...)
    state_colors = get_state_colors(n_states)
    n_features = len(feature_names)
    
    # Create figure: 3 rows
    # Row 1: Mean vectors (one subplot per state)
    # Row 2: Covariance matrices (one subplot per state)
    # Row 3: Transition matrix + start probs + model info
    
    fig = plt.figure(figsize=(3 * n_states + 2, 12))
    
    # Use GridSpec for flexible layout
    gs = fig.add_gridspec(3, n_states + 1, height_ratios=[1, 1.2, 1], 
                          width_ratios=[1] * n_states + [0.4], hspace=0.35, wspace=0.3)
    
    # =========================================================================
    # Row 1: Mean vectors μ_k for each state (k is new remapped state index)
    # =========================================================================
    for k in range(n_states):
        ax = fig.add_subplot(gs[0, k])
        
        y_pos = np.arange(n_features)
        ax.barh(y_pos, means[k], color=state_colors[k], edgecolor='black', alpha=0.8)
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(feature_names if k == 0 else [], fontsize=8)
        ax.set_xlabel('Mean value', fontsize=8)
        
        count = state_counts.get(k, state_counts.iloc[k] if k < len(state_counts) else 0)
        pct = count / state_counts.sum() * 100
        ax.set_title(f'State {k}\nμ_{k} ({pct:.1f}%)', fontsize=10, fontweight='bold', 
                    color=state_colors[k])
        style_axis(ax)
        ax.invert_yaxis()
    
    # Colorbar placeholder for row 1
    ax_cb1 = fig.add_subplot(gs[0, -1])
    ax_cb1.axis('off')
    ax_cb1.text(0.5, 0.5, 'Mean\nvectors\nμ_k', ha='center', va='center', fontsize=10, 
               transform=ax_cb1.transAxes)
    
    # =========================================================================
    # Row 2: Covariance matrices Σ_k for each state
    # =========================================================================
    # Find global min/max for consistent colorscale
    all_covars = covars.reshape(-1)
    vmax = np.percentile(np.abs(all_covars), 95)
    vmin = -vmax
    
    for k in range(n_states):
        ax = fig.add_subplot(gs[1, k])
        
        cov_k = covars[k]
        im = ax.imshow(cov_k, cmap='coolwarm', aspect='auto', vmin=vmin, vmax=vmax)
        
        ax.set_xticks(range(n_features))
        ax.set_yticks(range(n_features))
        ax.set_xticklabels([f[:3] for f in feature_names], fontsize=7, rotation=45, ha='right')
        ax.set_yticklabels([f[:3] for f in feature_names] if k == 0 else [], fontsize=7)
        
        ax.set_title(f'Σ_{k}', fontsize=10, fontweight='bold')
        
        # Add diagonal values as text (variances)
        for i in range(n_features):
            ax.text(i, i, f'{cov_k[i,i]:.2f}', ha='center', va='center', 
                   fontsize=6, fontweight='bold', color='black')
    
    # Colorbar for covariances
    ax_cb2 = fig.add_subplot(gs[1, -1])
    plt.colorbar(im, cax=ax_cb2, label='Covariance')
    
    # =========================================================================
    # Row 3: Transition matrix + start probs
    # =========================================================================
    # Transition matrix (takes most of the row)
    ax_trans = fig.add_subplot(gs[2, :n_states-1])
    
    im_trans = ax_trans.imshow(trans_matrix, cmap='Blues', vmin=0, vmax=1)
    ax_trans.set_xticks(range(n_states))
    ax_trans.set_yticks(range(n_states))
    ax_trans.set_xticklabels([f'S{i}' for i in range(n_states)])
    ax_trans.set_yticklabels([f'S{i}' for i in range(n_states)])
    ax_trans.set_xlabel('To state')
    ax_trans.set_ylabel('From state')
    ax_trans.set_title('Transition Matrix A[i,j] = P(s_t=j | s_{t-1}=i)', fontsize=10, fontweight='bold')
    
    for i in range(n_states):
        for j in range(n_states):
            color = 'black' if trans_matrix[i, j] < 0.5 else 'white'
            ax_trans.text(j, i, f'{trans_matrix[i, j]:.2f}', ha='center', va='center', 
                         color=color, fontsize=9)
    
    # Start probabilities
    ax_start = fig.add_subplot(gs[2, n_states-1])
    
    bars = ax_start.bar(range(n_states), startprob, color=state_colors, edgecolor='black')
    ax_start.set_xticks(range(n_states))
    ax_start.set_xticklabels([f'S{i}' for i in range(n_states)])
    ax_start.set_ylabel('Probability')
    ax_start.set_title('Start π', fontsize=10, fontweight='bold')
    ax_start.set_ylim(0, 1)
    style_axis(ax_start)
    
    for bar, p in zip(bars, startprob):
        if p > 0.05:
            ax_start.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                         f'{p:.2f}', ha='center', fontsize=8)
    
    # Model info text
    ax_info = fig.add_subplot(gs[2, -1])
    ax_info.axis('off')
    info_text = f"K={n_states}\n{results['covariance_type']}\ncov"
    ax_info.text(0.5, 0.5, info_text, ha='center', va='center', fontsize=10,
                transform=ax_info.transAxes, family='monospace')
    
    plt.suptitle(f'Gaussian HMM Model Parameters (K={n_states} states, {n_features} features)', 
                fontsize=14, fontweight='bold', y=0.98)
    
    plt.savefig(f"{viz_dir}/00_model_parameters.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print model summary
    print(f"\n{'='*80}")
    print(f"Gaussian HMM Model Structure (K={n_states} states)")
    print(f"{'='*80}")
    print(f"Each state k has an emission distribution: X | state=k ~ N(μ_k, Σ_k)")
    print(f"  • μ_k: mean vector of shape ({n_features},)")
    print(f"  • Σ_k: covariance matrix of shape ({n_features}, {n_features})")
    print(f"  • Covariance type: {results['covariance_type']}")
    print(f"\nTransition dynamics:")
    print(f"  • A[i,j] = P(state_t = j | state_{{t-1}} = i)")
    print(f"  • π[k] = P(state_0 = k) (start probability)")
    print(f"  • Note: States are reordered by occurrence (0=most common)")
    print(f"{'='*80}")


def plot_hmm_results(labels, results, feature_names, output_subdir='hmm_plots', max_points=10_000, random_state=RANDOM_SEED):
    """
    Visualize HMM results with consistent styling.
    
    Creates:
    1. Model selection plot (LL/AIC/BIC curves)
    2. Transition matrix heatmap
    3. State feature profiles (z-scored means) with RdYlGn colormap
    4. Per-feature z-score plots (one subplot per feature)
    5. State duration histograms (normal + log scale per state)
    6. State proportions pie chart
    7. Text summary of states
    
    Note: All state indices are already remapped (0=most common, etc.)
    
    IMPORTANT: 
    - Pie chart shows FRAME counts (total frames in each state)
    - Histogram shows BOUT/SEGMENT counts (number of continuous segments)
    """
    import os
    
    # Set seed for reproducibility in any random operations
    _set_all_seeds(random_state)
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = results['best_n']
    state_counts = results['state_counts']  # FRAME counts per state
    trans_matrix = results['transition_matrix']
    
    state_colors = get_state_colors(n_states)
    
    # =========================================================================
    # Compute segment/bout statistics FIRST (needed for multiple plots)
    # =========================================================================
    state_changes = np.diff(labels) != 0
    change_indices = np.where(state_changes)[0] + 1
    change_indices = np.concatenate([[0], change_indices, [len(labels)]])
    
    durations, duration_states = [], []
    for i in range(len(change_indices) - 1):
        start, end = change_indices[i], change_indices[i + 1]
        durations.append(end - start)
        duration_states.append(labels[start])
    
    durations = np.array(durations)
    duration_states = np.array(duration_states)
    
    # Compute segment counts per state
    segment_counts = {s: (duration_states == s).sum() for s in range(n_states)}
    total_segments = len(durations)
    total_frames = len(labels)
    
    # =========================================================================
    # 1. Model selection plot (LL/AIC/BIC)
    # =========================================================================
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    n_range = results['n_components_range']
    
    plot_configs = [
        (results['log_likelihoods'], 'b', 'Log-Likelihood', 'Log-Likelihood (higher is better)'),
        (results['aics'], 'g', 'AIC', 'AIC (lower is better)'),
        (results['bics'], 'm', 'BIC', 'BIC (lower is better)'),
    ]
    
    for ax, (data, color, ylabel, title) in zip(axes[:3], plot_configs):
        ax.plot(n_range, data, f'{color}-o')
        ax.axvline(results['best_n'], color='r', linestyle='--', label=f'Selected: {results["best_n"]}')
        ax.set_xlabel('Number of states')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend()
        style_axis(ax)
    
    axes[3].errorbar(n_range, results['cv_scores'], yerr=results.get('cv_stds', None), 
                     fmt='c-o', capsize=3, capthick=1)
    axes[3].axvline(results['best_n'], color='r', linestyle='--')
    axes[3].set_xlabel('Number of states')
    axes[3].set_ylabel('CV Log-Likelihood')
    axes[3].set_title('Cross-Validation LL (higher is better)')
    style_axis(axes[3])
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/01_model_selection.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 2. Transition matrix heatmap
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    im = ax.imshow(trans_matrix, cmap='Blues', vmin=0, vmax=1)
    # print the trans_matrix in a nice way
    for i in range(n_states):
        for j in range(n_states):   
            print(f"From state {i} to state {j}: P={trans_matrix[i, j]:.2f}")
    ax.set_xticks(range(n_states))
    ax.set_yticks(range(n_states))
    ax.set_xlabel('To state')
    ax.set_ylabel('From state')
    ax.set_title('State Transition Probabilities (reordered by occurrence)')
    
    for i in range(n_states):
        for j in range(n_states):
            color = 'black' if trans_matrix[i, j] < 0.5 else 'white'
            ax.text(j, i, f'{trans_matrix[i, j]:.2f}', ha='center', va='center', color=color)
    
    plt.colorbar(im, ax=ax, label='Probability')
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/02_transition_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 3. State feature profiles (z-scored means) - RdYlGn colormap
    # =========================================================================
    means = results['means']  # Already remapped by state occurrence
    means_zscore = (means - means.mean(axis=0)) / (means.std(axis=0) + 1e-8)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    im = ax.imshow(means_zscore.T, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)
    ax.set_xticks(range(n_states))
    ax.set_yticks(range(len(feature_names)))
    ax.set_yticklabels(feature_names)
    ax.set_xlabel('State (0=most common)')
    ax.set_ylabel('Feature')
    ax.set_title('State Feature Profiles (z-scored means, reordered by occurrence)')
    
    for i in range(len(feature_names)):
        for j in range(n_states):
            val = means_zscore[j, i]
            text_color = 'black' if -1 < val < 1 else 'white'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                   color=text_color, fontsize=9, fontweight='bold')
    
    plt.colorbar(im, ax=ax, label='Z-score')
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/03_feature_profiles.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 4. Per-feature z-score plots (5 columns)
    # =========================================================================
    n_features = len(feature_names)
    n_cols = 5
    n_rows = int(np.ceil(n_features / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
    axes = np.atleast_2d(axes)
    
    for feat_idx, feat_name in enumerate(feature_names):
        row, col = feat_idx // n_cols, feat_idx % n_cols
        ax = axes[row, col]
        
        zscores = means_zscore[:, feat_idx]
        y_positions = np.arange(n_states)
        
        bars = ax.barh(y_positions, zscores, color=state_colors, edgecolor='black', height=0.7)
        
        ax.set_yticks(y_positions)
        ax.set_yticklabels([f'State {s}' for s in range(n_states)])
        ax.set_xlabel('Z-score')
        ax.set_title(feat_name, fontweight='bold')
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlim(-3, 3)
        style_axis(ax)
        
        for bar, zscore in zip(bars, zscores):
            x_pos = bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1
            ha = 'left' if bar.get_width() >= 0 else 'right'
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{zscore:.2f}', 
                   va='center', ha=ha, fontsize=8)
    
    for idx in range(n_features, n_rows * n_cols):
        axes[idx // n_cols, idx % n_cols].set_visible(False)
    
    plt.suptitle('Feature Z-scores by State (0=most common)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/04a_feature_zscores_per_feature.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 4b. Per-state z-score plots (one subplot per state, showing all features)
    #     Uses red/green coloring based on z-score sign (like RdYlGn)
    # =========================================================================
    n_cols_state = min(n_states, 4)  # Max 4 columns
    n_rows_state = int(np.ceil(n_states / n_cols_state))
    
    fig, axes = plt.subplots(n_rows_state, n_cols_state, figsize=(5 * n_cols_state, 3 * n_rows_state))
    axes = np.atleast_2d(axes)
    if n_rows_state == 1:
        axes = axes.reshape(1, -1)
    
    # RdYlGn-inspired colors: green for positive, red for negative
    def zscore_to_color(z):
        """Map z-score to RdYlGn-like color: negative=red, positive=green."""
        # Normalize to [-2, 2] range, clip extremes
        z_clipped = np.clip(z, -2, 2)
        # Map to [0, 1]: -2 -> 0 (red), 0 -> 0.5 (yellow), 2 -> 1 (green)
        norm_val = (z_clipped + 2) / 4
        cmap = plt.cm.RdYlGn
        return cmap(norm_val)
    
    for state_idx in range(n_states):
        row, col = state_idx // n_cols_state, state_idx % n_cols_state
        ax = axes[row, col]
        
        zscores = means_zscore[state_idx, :]  # All features for this state
        y_positions = np.arange(n_features)
        
        # Color each bar based on its z-score value
        bar_colors = [zscore_to_color(z) for z in zscores]
        
        bars = ax.barh(y_positions, zscores, color=bar_colors, edgecolor='black', height=0.7)
        
        ax.set_yticks(y_positions)
        ax.set_yticklabels(feature_names, fontsize=9)
        ax.set_xlabel('Z-score')
        
        # Get state count for title
        frame_count = state_counts.get(state_idx, state_counts.iloc[state_idx] if state_idx < len(state_counts) else 0)
        pct = frame_count / total_frames * 100
        ax.set_title(f'State {state_idx} ({pct:.1f}%)', fontweight='bold', color=state_colors[state_idx])
        
        ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xlim(-3, 3)
        ax.invert_yaxis()  # Top-to-bottom reading order
        style_axis(ax)
        
        # Add z-score labels
        for bar, zscore in zip(bars, zscores):
            x_pos = bar.get_width() + 0.1 if bar.get_width() >= 0 else bar.get_width() - 0.1
            ha = 'left' if bar.get_width() >= 0 else 'right'
            ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{zscore:.2f}', 
                   va='center', ha=ha, fontsize=8)
    
    # Hide unused subplots
    for idx in range(n_states, n_rows_state * n_cols_state):
        axes[idx // n_cols_state, idx % n_cols_state].set_visible(False)
    
    plt.suptitle('Feature Z-scores by State (red=negative, green=positive)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/04b_feature_zscores_per_state.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 5. State duration histograms (normal + log scale per state)
    #    NOTE: n here = number of SEGMENTS/BOUTS, not frames!
    # =========================================================================
    fig, axes = plt.subplots(2, n_states, figsize=(3.5 * n_states, 7))
    if n_states == 1:
        axes = axes.reshape(2, 1)
    
    for state in range(n_states):
        state_durations = durations[duration_states == state]
        n_segments = len(state_durations)
        n_frames = state_counts.get(state, state_counts.iloc[state] if state < len(state_counts) else 0)
        
        if n_segments == 0:
            for row in range(2):
                axes[row, state].text(0.5, 0.5, 'No data', ha='center', va='center', 
                                      transform=axes[row, state].transAxes)
            continue
        
        median_dur, mean_dur = np.median(state_durations), np.mean(state_durations)
        
        for row, (yscale, ylabel, show_legend) in enumerate([
            ('linear', 'Count', True), ('log', 'Count (log)', False)
        ]):
            ax = axes[row, state]
            ax.hist(state_durations, bins=30, color=state_colors[state], 
                   edgecolor='black', alpha=0.7)
            ax.axvline(median_dur, color='red', linestyle='--', linewidth=1.5, 
                      label=f'Median: {median_dur:.0f}')
            ax.axvline(mean_dur, color='blue', linestyle=':', linewidth=1.5, 
                      label=f'Mean: {mean_dur:.1f}')
            ax.set_xlabel('Duration (frames)')
            ax.set_ylabel(ylabel)
            ax.set_yscale(yscale)
            title_suffix = '' if row == 0 else ' (log scale)'
            # Clarify: n_bouts = segments, n_frames = total frames
            ax.set_title(f'State {state}\n({n_segments} bouts, {n_frames:,} frames){title_suffix}', 
                        fontweight='bold', fontsize=9)
            style_axis(ax)
            if show_legend:
                ax.legend(fontsize=8)
    
    plt.suptitle(f'State Duration Distributions ({total_segments} total bouts, {total_frames:,} total frames)', 
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/05_duration_histogram_by_state.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 6. State proportions pie chart - FRAME counts (not segment counts)
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    
    proportions = state_counts / state_counts.sum()
    
    # Use actual state indices from state_counts (in case some states are missing)
    actual_states = list(state_counts.index) if hasattr(state_counts, 'index') else list(range(len(state_counts)))
    n_pie_states = len(proportions)
    pie_colors = [state_colors[s] if s < len(state_colors) else state_colors[s % len(state_colors)] for s in actual_states]
    
    # Use actual frame counts from state_counts
    wedges, texts, autotexts = ax.pie(
        proportions.values, 
        labels=[f'State {s}' for s in actual_states],
        colors=pie_colors,
        autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * total_frames):,} frames)',
        startangle=90,
        explode=[0.02] * n_pie_states,
        wedgeprops={'edgecolor': 'black', 'linewidth': 1}
    )
    
    for autotext in autotexts:
        autotext.set_fontsize(9)
        autotext.set_fontweight('bold')
    
    ax.set_title(f'State Occupancy - FRAMES (total: {total_frames:,})\n(0=most common)', 
                fontsize=14, fontweight='bold')
    ax.axis('equal')
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/06_state_proportions.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 7. State switch rate donut chart
    # =========================================================================
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    prop_switches = results['switch_rate']
    switch_data = [prop_switches, 1 - prop_switches]
    donut_colors = ['#e74c3c', '#2ecc71']  # Red for switch, green for stay
    pie_labels = [f'Switch\n({prop_switches:.1%})', f'Stay\n({1-prop_switches:.1%})']
    
    wedges, texts, autotexts = ax.pie(
        switch_data, 
        colors=donut_colors, 
        labels=pie_labels,
        autopct='', 
        startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    
    # Add center circle for donut effect
    centre_circle = plt.Circle((0, 0), 0.5, fc='white')
    ax.add_artist(centre_circle)
    
    # Center text
    ax.text(0, 0, f'Switch Rate\n{prop_switches:.1%}', ha='center', va='center', 
            fontsize=14, fontweight='bold')
    ax.set_title(f'HMM State Transitions\n(Frame-to-Frame)', fontsize=12, fontweight='bold')
    ax.axis('equal')
    
    plt.tight_layout()
    plt.savefig(f"{viz_dir}/07_switch_rate.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    # =========================================================================
    # 8. State summary statistics (text)
    # =========================================================================
    state_profiles = pd.DataFrame(means_zscore, columns=feature_names)
    
    print(f"\n{'='*130}")
    print(f"=== HMM State Summary (n_states={n_states}) - States Ordered by Occurrence (0=most common) ===")
    print(f"{'='*130}")
    print(f"{'State':<8} {'Frames':<12} {'%':<8} {'Bouts':<10} {'Avg Dur':<10} {'Top 3 HIGH features':<40} {'Top 3 LOW features':<40}")
    print(f"{'-'*130}")
    
    for s in range(n_states):
        frame_count = state_counts.get(s, state_counts.iloc[s] if s < len(state_counts) else 0)
        pct = frame_count / total_frames * 100
        bout_count = segment_counts[s]
        avg_dur = frame_count / bout_count if bout_count > 0 else 0
        
        z_scores = state_profiles.iloc[s].sort_values()
        top_high = ', '.join([f"{f}(+{z:.1f})" for f, z in z_scores.tail(3)[::-1].items()])
        top_low = ', '.join([f"{f}({z:.1f})" for f, z in z_scores.head(3).items()])
        
        print(f"S{s:<7} {frame_count:<12,} {pct:>5.1f}%   {bout_count:<10,} {avg_dur:<10.1f} {top_high:<40} {top_low:<40}")
    
    print(f"{'-'*130}")
    print(f"Total: {total_frames:,} frames, {total_segments:,} bouts")
    print(f"Switch rate: {results['switch_rate']:.3f} (fraction of frames with state change)")
    print(f"Random state used: {results.get('random_state', 'N/A')}")
    print(f"{'='*130}")
    
    print(f"\n✓ Saved HMM visualization plots to {viz_dir}")


def plot_hmm_states_on_embedding(embedding, labels, title_prefix="", output_subdir='hmm_plots', max_points=10_000, random_state=RANDOM_SEED):
    """Plot HMM states on a 2D or 3D embedding.
    
    Note: States are already remapped so state 0 = most common, etc.
    """
    import os
    
    # Set seed for reproducible subsampling
    _set_all_seeds(random_state)
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = len(np.unique(labels))
    state_colors = get_state_colors(n_states)
    
    if len(labels) > max_points:
        idx = np.sort(np.random.choice(len(labels), max_points, replace=False))
        embedding_sub, labels_sub = embedding[idx], labels[idx]
    else:
        embedding_sub, labels_sub = embedding, labels
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    for state in range(n_states):
        mask = labels_sub == state
        ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], 
                  s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
    
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_title(f'{title_prefix} - HMM States (n={n_states}, 0=most common)')
    ax.legend(markerscale=5)
    style_axis(ax)
    
    plt.tight_layout()
    safe_title = title_prefix.lower().replace(' ', '_')
    plt.savefig(f"{viz_dir}/07_{safe_title}_states_2d.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    if embedding.shape[1] >= 3:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        for state in range(n_states):
            mask = labels_sub == state
            ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], embedding_sub[mask, 2],
                      s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
        
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
        ax.set_zlabel('Dimension 3')
        ax.set_title(f'{title_prefix} - HMM States 3D (n={n_states}, 0=most common)')
        ax.legend(markerscale=5)
        
        plt.tight_layout()
        plt.savefig(f"{viz_dir}/07_{safe_title}_states_3d.png", dpi=150, bbox_inches='tight')
        plt.show()
    
    print(f"✓ Saved embedding plots to {viz_dir}")

In [ ]:
# trial_lengths = framew_beh.loc[actions.index].reset_index().groupby(('session_id', 'trial_id')).size()
# trial_lengths
# valid_trials_mask = framew_beh.trial_id.notna()
lengths = framew_beh.loc[actions.index].reset_index().groupby(['session_id', 'trial_id']).size()

# actions.index
# framew_beh.loc[actions.index].reset_index()[['session_id', 'trial_id']]
lengths

In [ ]:
# from sklearn.preprocessing import StandardScaler

# Standardize the input features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(actions.values)
# print(X_scaled)

# Fit all HMM models
fitting_results = fit_hmm_models(
    actions_scaled.values, 
    lengths=lengths.tolist(),
    n_components_range=range(4, 13), 
    covariance_type='full',
    n_iter=100,
    
)
print(fitting_results)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(actions_scaled, fitting_results, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)

In [ ]:
# from sklearn.preprocessing import StandardScaler

# Standardize the input features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(actions.values)
# print(X_scaled)

# Fit all HMM models
fitting_results = fit_hmm_models(
    actions_scaled.values, 
    lengths=lengths.tolist(),
    n_components_range=range(4, 13), 
    covariance_type='full',
    n_iter=100,
    
)
print(fitting_results)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(actions_scaled, fitting_results, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)


State	%	Bouts	Avg Dur	Interpretation
S4	46.7%	19,831	14.8	Active forward locomotion — High forward velocity & acceleration; moderate rotation. Longest sustained bouts. Dominant behavioral mode.
S0	30.3%	19,056	10.0	Rotational movement — High rotation velocity; low forward acceleration. Turning/orienting behavior. Second most common state.
S1	12.9%	4,125	19.7	Quiet/stationary — All velocities & accelerations near zero or negative. Longest avg bout duration (20 frames). Resting/pausing state.
S8	4.3%	8,380	3.2	Licking + deceleration — High licking, negative forward acc. Short bouts (3.2). Reward consumption with slowing.
S3	3.5%	7,729	2.8	Licking + brief acceleration — High licking with forward acc pulse. Very short (2.8). Anticipatory licking during approach.
S5	2.0%	1,685	7.6	Strong rotational deceleration — Very low rotation/rotation acc. Mid-length bouts. Stopping rotational movement.
S2	0.3%	878	2.1	Licking while still — High licking, low all velocities. Shortest bouts (2.1). Stationary licking.
S7	0.1%	137	3.8	Acceleration burst — High rotation & forward acc. Rare, short. Movement initiation/startle.
S6	0.1%	0	—	Artifact/transient — 525 frames but 0 detected bouts (all within session boundaries?). High velocity, low acc. Edge case.

In [ ]:
# from sklearn.preprocessing import StandardScaler

# Standardize the input features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(actions.values)
# print(X_scaled)

# Fit all HMM models
fitting_results = fit_hmm_models(
    actions_scaled.values, 
    lengths=lengths.tolist(),
    n_components_range=range(4, 13), 
    covariance_type='full',
    n_iter=100,
    
)
print(fitting_results)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(actions_scaled, fitting_results, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)

In [ ]:
# from sklearn.preprocessing import StandardScaler

# Standardize the input features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(actions.values)

# Fit all HMM models
fitting_results = fit_hmm_models(
    X_scaled, 
    lengths=lengths.tolist(),
    n_components_range=range(3, 11), 
    covariance_type='spherical',
    n_iter=300
)
print(fitting_results)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(X_scaled, fitting_results, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)

In [ ]:
# diag

from sklearn.preprocessing import StandardScaler

# Standardize the input features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(actions.values)

# Fit all HMM models
fitting_results = fit_hmm_models(
    X_scaled,
    lengths=lengths.tolist(),
    n_components_range=range(3, 11), 
    covariance_type='diag',
    n_iter=300
)
print(fitting_results)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(X_scaled, fitting_results, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)

In [ ]:
hmm_labels

In [ ]:
sph_hmm_labels = pd.Series(hmm_labels, index=actions.index, name='spherical_hmm_state')
sph_hmm_labels

In [ ]:

def plot_one_trial(trial_data, s_id, trial_id, cluster_labels=None, n_clusters=None):
    """
    Plot a single trial with behavioral metrics and events.
    
    Parameters:
    -----------
    trial_data : pd.DataFrame
        Trial data with time index
    s_id : str
        Session ID
    trial_id : int
        Trial ID
    cluster_labels : array-like, optional
        GMM cluster assignments for each frame (same length as trial_data)
    n_clusters : int, optional
        Number of clusters (required if cluster_labels provided)
    """
    
    # Create figure with optional cluster subplot
    if cluster_labels is not None:
        fig, (ax_clusters, ax_main) = plt.subplots(
            2, 1, figsize=(13, 6), 
            height_ratios=[1, 4],
            sharex=True
        )
        plt.subplots_adjust(hspace=0.05)
    else:
        fig, ax_main = plt.subplots(figsize=(9, 3))

    # main_metric = 'frame_velocity'
    # main_metric = 'frame_raw'
    main_metric = 'frame_RawYawPitch_abs_vel_sum'

    # which cue was shown
    color = 'orange' if trial_data['cue'].iloc[0] == 1 else 'purple'

    # cue limits
    cue_entry_time = trial_data[(trial_data['frame_position'] >-80) & 
                                (trial_data['frame_position'] <25)].index.values
    ax_main.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.3)

    # R1 limits
    R1_entry_time = trial_data[trial_data['track_zone']=='reward1Zone'].index.values
    ax_main.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.2)
    
    # R2 limits
    R2_entry_time = trial_data[trial_data['track_zone']=='reward2Zone'].index.values
    ax_main.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.5)

    r1_thr = (trial_data['velocity_threshold_at_R1'] * trial_data['fps'])
    r1_frames = r1_thr[(r1_thr.index>R1_entry_time[0]) & (r1_thr.index<R1_entry_time[-1])].index
    ax_main.plot(r1_thr, linestyle='--', color='black', alpha=.4, label='R1 thr.')
    
    r2_thr = (trial_data['velocity_threshold_at_R2'] * trial_data['fps'])
    r2_frames = r2_thr[(r2_thr.index>R2_entry_time[0]) & (r2_thr.index<R2_entry_time[-1])].index
    ax_main.plot(r2_thr.loc[r2_frames], linestyle='--', color='black', alpha=.4, label='R2 thr.')

    # events
    event_labels = {
        'reward-sound_count': {'label': 'Sound', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': '>'},
        'reward-valve-open_count': {'label': 'Reward', 'color': 'green', 'size': 80, 'alpha': 0.7, 'marker': 'o'},
        'lick_count': {'label': 'Lick', 'color': 'black', 'size': 50, 'alpha': 0.4, 'marker': '|'},
        'reward-removed_count': {'label': 'Reward-removed', 'color': 'red', 'size': 80, 'alpha': 0.7, 'marker': 'x'}
    }
    
    for event in event_labels.keys():
        event_data = trial_data[trial_data[event] >= 1]
        if len(event_data) == 0:
            continue
        
        ax_main.scatter(event_data.index, event_data[main_metric],
                       color=event_labels[event]['color'], s=event_labels[event]['size'],
                       marker=event_labels[event]['marker'], label=event_labels[event]['label'],
                       alpha=event_labels[event]['alpha'], zorder=5)

    renamer = {
        'frame_raw_500msMedian': 'ball_forward_vel',
        'frame_YawPitch_abs_vel_500msMedian': 'ball_rotation_vel',
        'frame_RawYawPitch_abs_vel_sum_500msMedian': 'ball_forward+rotation_vel',
        'frame_RawYawPitch_abs_acc_sum_500msMedian': 'ball_forward+rotation_acc',
        'frame_raw_acc_500msMedian': 'ball_forward_acc',
        'frame_YawPitch_abs_acc_500msMedian': 'ball_rotation_acc',
        'lick_detected': 'licking',
    }

    # main metrics
    ax_main.plot(trial_data[main_metric], zorder=1, color='blue', label='ball-summed')
    ax_main.plot(trial_data['frame_RawYawPitch_abs_vel_sum_500msMedian'], zorder=1, color='darkblue', alpha=0.5, linestyle='-', label='ball-summed=smoothed')
    # ax_main.plot(trial_data['frame_pitch'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-sideways')
    ax_main.plot(trial_data['frame_YawPitch_abs_vel_500msMedian'].abs(), zorder=1, color='purple', alpha=0.3, linestyle=':', label='ball-rotation-smoothed')
    ax_main.plot(trial_data['frame_YawPitch_abs_acc_500msMedian'], zorder=1, color='purple', alpha=0.4, linestyle=':', label='ball-rotation-sm-acc')
    
    ax_main.plot(trial_data['frame_raw_500msMedian'].abs(), zorder=1, color='blue', alpha=0.3, linestyle=':', label='ball-forward-smoothed')
    ax_main.plot(trial_data['frame_raw_acc_500msMedian'], zorder=1, color='blue', alpha=0.4, linestyle=':', label='ball-forward-sm-acc')
    # ax_main.plot(trial_data['facecam_pose_nose_neck_body1_angle'].rolling(6).median(), zorder=1, color='darkred', linestyle=':', label='head-angle')

    ax_main.set_xlabel('Time (s)')
    ax_main.set_ylabel('Velocity [cm/s]')
    ax_main.set_title(f'Session {s_id} - Trial {trial_id}')
    ax_main.legend(ncol=2, fontsize=8)
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)

    # =========================================================================
    # Cluster visualization subplot (if labels provided)
    # =========================================================================
    if cluster_labels is not None:
        times = trial_data.index.values
        cluster_colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))
        
        # Use scatter plot - one point per frame, colored by cluster
        for c in range(-1, n_clusters):
            mask = cluster_labels == c
            if not mask.any():
                continue
            
            if c == -1:
                color = 'lightgray'
                y_val = -0.3
            else:
                color = cluster_colors[c]
                y_val = c
            
            ax_clusters.scatter(
                times[mask], 
                np.full(mask.sum(), y_val),
                c=[color], 
                s=200,
                marker='|',
                linewidths=1,
            )
        
        # Format cluster axis
        ax_clusters.set_ylim(-0.5, n_clusters - 0.5)
        ax_clusters.set_yticks(range(n_clusters))
        ax_clusters.set_yticklabels([f'C{c}' for c in range(n_clusters)], fontsize=8)
        ax_clusters.set_ylabel('Cluster', fontsize=9)
        ax_clusters.spines['top'].set_visible(False)
        ax_clusters.spines['right'].set_visible(False)
        ax_clusters.spines['bottom'].set_visible(False)
        ax_clusters.tick_params(bottom=False)
        
        # Add zone shading to cluster plot too
        ax_clusters.axvspan(cue_entry_time[0], cue_entry_time[-1], color=color, alpha=0.2)
        ax_clusters.axvspan(R1_entry_time[0], R1_entry_time[-1], color='grey', alpha=0.1)
        ax_clusters.axvspan(R2_entry_time[0], R2_entry_time[-1], color='darkgrey', alpha=0.2)

    plt.tight_layout()
    # plt.show()
    return fig

In [ ]:
plt.close('all')

s_id = 0
s_id = framew_beh.index.unique('session_id')[s_id]
time_col = 'frame_pc_timestamp'

session_data = framew_beh.loc[framew_beh.index.get_level_values('session_id') == s_id]
for trial_id in session_data['trial_id'].dropna().unique().astype(int):
    if trial_id == -1:
        continue
    trial_data = session_data.loc[session_data['trial_id'] == trial_id]
    
    trial_frame_ids = trial_data.index.values
    trial_data = trial_data.set_index(time_col, drop=False, append=False)
    trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    
    
    trial_clust_assign = sph_hmm_labels.reindex(trial_frame_ids)#.fillna(-1).astype({'hmm_k9': int})
    print(trial_clust_assign.shape, sph_hmm_labels.index.isin(trial_frame_ids).sum())
    print(trial_clust_assign)
    fig = plot_one_trial(trial_data, s_id, trial_id, cluster_labels=trial_clust_assign.values, n_clusters=9)
    plt.show()
    
    if trial_id == 4:
        break  


In [ ]:
plt.close('all')

s_id = 0
s_id = framew_beh.index.unique('session_id')[s_id]
time_col = 'frame_pc_timestamp'
timewindow = (-60, 60)  # ± 1 second around R1 entry

session_data = framew_beh.loc[framew_beh.index.get_level_values('session_id') == s_id]
for trial_id in session_data['trial_id'].dropna().unique().astype(int):
    if trial_id == -1:
        continue
    trial_data = session_data.loc[session_data['trial_id'] == trial_id]
    
    trial_frame_ids = trial_data.index.values
    R1_entry_frame_i = np.argwhere(trial_data['track_zone']=='reward1Zone')[0][0]
    
    # trial_data = trial_data.set_index(time_col, drop=False, append=False)
    # trial_data.index = (trial_data.index - trial_data.index[0]) /1e6
    
    outcome = trial_data['trial_outcome'].iloc[0].astype(bool)
    print(outcome)
    trial_clust_assign = sph_hmm_labels.reindex(trial_frame_ids).iloc[R1_entry_frame_i+timewindow[0]:R1_entry_frame_i+timewindow[1]].fillna(-1).astype(int)
    print(R1_entry_frame_i)
    print(trial_clust_assign)
    # print(trial_clust_assign.iloc[])
    
    trial_clust_assign = trial_clust_assign.rolling(10, center=True, min_periods=1).median()
    
    plt.plot(trial_clust_assign.values, color='green' if outcome else 'red', alpha=0.3)
    plt.axvline(-timewindow[0], color='red', linestyle='--')
    plt.title(f'Session {s_id} - Trial {trial_id} - Outcome: {"Success" if outcome else "Failure"}')
    plt.xlabel('Frames relative to R1 entry')
    plt.ylabel('HMM Cluster Assignment')
    
    # break
    # fig = plot_one_trial(trial_data, s_id, trial_id, cluster_labels=trial_clust_assign.values, n_clusters=9)
    # plt.show()
    
    # if trial_id == 4:
    #     break  

plt.show()

In [ ]:

# # Fit all HMM models
fitting_results_diag = fit_hmm_models(
    # X_scaled,
    actions_scaled.values, 
    lengths=lengths.values,
    n_components_range=range(4, 15), 
    covariance_type='full',
    n_iter=100,
    n_init=8,
    sample_n_trials=600,
)

# Select best model (by BIC) and decode states sorted by occurrence
hmm_labels, hmm_results = select_and_decode_hmm(actions_scaled.values, 
                                                fitting_results_diag, best_idx=None)

# Visualize model parameters (means, covariances, transitions)
plot_model_parameters(hmm_results, feature_names=actions.columns.tolist())

# Visualize all results
plot_hmm_results(
    hmm_labels, hmm_results, 
    feature_names=actions.columns.tolist(),
    output_subdir='hmm_plots'
)


In [ ]:
def plot_hmm_states_on_embedding(embedding, labels, title_prefix="", output_subdir='hmm_plots', max_points=10_000, random_state=RANDOM_SEED):
    """Plot HMM states on a 2D or 3D embedding."""
    import os
    
    # Set seed for reproducible subsampling
    _set_all_seeds(random_state)
    
    viz_dir = f"{output_dir}/{output_subdir}/"
    os.makedirs(viz_dir, exist_ok=True)
    
    n_states = len(np.unique(labels))
    state_colors = get_state_colors(n_states)
    
    if len(labels) > max_points:
        idx = np.sort(np.random.choice(len(labels), max_points, replace=False))
        embedding_sub, labels_sub = embedding[idx], labels[idx]
    else:
        embedding_sub, labels_sub = embedding, labels
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    for state in range(n_states):
        mask = labels_sub == state
        ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], 
                  s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
    
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_title(f'{title_prefix} - HMM States (n={n_states})')
    ax.legend(markerscale=5)
    style_axis(ax)
    
    plt.tight_layout()
    safe_title = title_prefix.lower().replace(' ', '_')
    plt.savefig(f"{viz_dir}/07_{safe_title}_states_2d.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    if embedding.shape[1] >= 3:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        for state in range(n_states):
            mask = labels_sub == state
            ax.scatter(embedding_sub[mask, 0], embedding_sub[mask, 1], embedding_sub[mask, 2],
                      s=1, alpha=0.5, label=f'State {state}', color=state_colors[state])
        
        ax.set_xlabel('Dimension 1')
        ax.set_ylabel('Dimension 2')
        ax.set_zlabel('Dimension 3')
        ax.set_title(f'{title_prefix} - HMM States 3D (n={n_states})')
        ax.legend(markerscale=5)
        
        plt.tight_layout()
        plt.savefig(f"{viz_dir}/07_{safe_title}_states_3d.png", dpi=150, bbox_inches='tight')
        plt.show()
    
    print(f"✓ Saved embedding plots to {viz_dir}")